In [16]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip -q install neo4j pandas python-dotenv

In [17]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import neo4j
import pandas as pd
import os

load_dotenv()

True

In [18]:
load_dotenv()
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()


In [19]:
def run_cypher(query: str, params: dict | None = None) -> pd.DataFrame:
    params = params or {}
    with driver.session() as session:
        res = session.run(query, params)
        rows = [r.data() for r in res]
    return pd.DataFrame(rows)

def run_cypher_many(queries: list[str]) -> None:
    with driver.session() as session:
        for query in queries:
            session.run(query).consume()

### Create Constraints For Multiple CSV Conflicts

In [20]:
queries = [
    "CREATE CONSTRAINT person_id IF NOT EXISTS FOR (n:Person) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT location_id IF NOT EXISTS FOR (n:Location) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT event_id IF NOT EXISTS FOR (n:Event) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT claim_id IF NOT EXISTS FOR (n:Claim) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT evidence_id IF NOT EXISTS FOR (n:Evidence) REQUIRE n.id IS UNIQUE",
]

run_cypher_many(queries)

### Load Each CSV By Label

In [23]:
def load_csv(filename: str) -> None:
    person_q = f"""
        LOAD CSV WITH HEADERS FROM 'file:///{filename}' AS row
        WITH row WHERE row._labels = ':Person'
        MERGE (n:Person {{id: row.id}})
        SET n.name = row.name,
            n.type = row.type,
            n.status = row.status,
            n.dob = row.dob;
    """
    location_q = f"""
        LOAD CSV WITH HEADERS FROM 'file:///{filename}' AS row
        WITH row WHERE row._labels = ':Location'
        MERGE (n:Location {{id: row.id}})
        SET n.name = row.name,
            n.city = row.city,
            n.address = row.address,
            n.type = row.type;
    """
    case_q = f"""
        LOAD CSV WITH HEADERS FROM 'file:///{filename}' AS row
        WITH row WHERE row._labels = ':Case'
        MERGE (n:Case {{id: row.id}})
        SET n.name = row.name,
            n.status = row.status,
            n.dateOpened = row.dateOpened;
    """
    event_q = f"""
        LOAD CSV WITH HEADERS FROM 'file:///{filename}' AS row
        WITH row WHERE row._labels = ':Event'
        MERGE (n:Event {{id: row.id}})
        SET n.name = row.name,
            n.type = row.type,
            n.date = row.date,
            n.description = row.description;
    """
    evidence_q = f"""
        LOAD CSV WITH HEADERS FROM 'file:///{filename}' AS row
        WITH row WHERE row._labels = ':Evidence'
        MERGE (n:Evidence {{id: row.id}})
        SET n.name = row.name,
            n.type = row.type,
            n.status = row.status,
            n.description = row.description;
    """
    queries = [
        person_q,
        location_q,
        case_q,
        event_q,
        evidence_q
    ]
    run_cypher_many(queries)


### CSV FIles

In [ ]:
"""
    Make sure to upload csv files to docker import folder
    Can be done with docker cp <SOURCE FILE> <container id>:import/<DESTINATION NAME>
"""
csvs = [
    "cold_case2.csv",
    "cold_case3.csv",
    "cold_case4.csv",
    "cold_case5.csv",
    "cold_case6.csv"
]

for csv in csvs:
    load_csv(csv)

### Load Relationships From GraphML Files

In [26]:
import xml.etree.ElementTree as ET
import re
from neo4j import GraphDatabase
from collections import defaultdict
from pathlib import Path

NS = {"g": "http://graphml.graphdrawing.org/xmlns"}


def sanitize_rel_type(rel_type: str) -> str:
    if not rel_type:
        raise ValueError("Empty relationship type")

    rel_type = rel_type.upper()
    rel_type = re.sub(r"[^A-Z0-9_]", "_", rel_type)

    if not rel_type[0].isalpha():
        rel_type = "R_" + rel_type

    return rel_type

def import_graphml_relationships(graphml_file: str) -> None:
    graphml_path = Path(graphml_file)

    if graphml_path.suffix.lower() != ".graphml":
        graphml_path = graphml_path.with_suffix(".graphml")

    if not graphml_path.exists():
        raise FileNotFoundError(f"GraphML file not found: {graphml_path}")

    tree = ET.parse(graphml_path)
    root = tree.getroot()

    key_map = {}
    for k in root.findall("g:key", NS):
        key_id = k.attrib.get("id")
        attr_name = k.attrib.get("attr.name")
        key_for = k.attrib.get("for")
        key_map[(key_for, key_id)] = attr_name

    node_id_map = {}
    for node in root.findall(".//g:node", NS):
        graphml_node_id = node.attrib["id"]
        props = {}

        for data in node.findall("g:data", NS):
            key_id = data.attrib["key"]
            attr_name = key_map.get(("node", key_id), key_id)
            props[attr_name] = data.text

        if "id" in props and props["id"]:
            node_id_map[graphml_node_id] = props["id"]

    rels_by_type = defaultdict(list)

    for edge in root.findall(".//g:edge", NS):
        source_id = node_id_map.get(edge.attrib["source"])
        target_id = node_id_map.get(edge.attrib["target"])

        if not source_id or not target_id:
            continue

        rel_type = edge.attrib.get("label")

        if not rel_type:
            for data in edge.findall("g:data", NS):
                key_id = data.attrib["key"]
                attr_name = key_map.get(("edge", key_id), key_id)
                if attr_name == "type":
                    rel_type = data.text
                    break

        if not rel_type:
            continue

        safe_rel_type = sanitize_rel_type(rel_type)

        props = {}
        for data in edge.findall("g:data", NS):
            key_id = data.attrib["key"]
            attr_name = key_map.get(("edge", key_id), key_id)
            props[attr_name] = data.text

        rels_by_type[safe_rel_type].append({
            "source_id": source_id,
            "target_id": target_id,
            "props": props,
        })

    total = sum(len(v) for v in rels_by_type.values())
    print(f"Parsed {total} relationships from {graphml_path.name}")

    with driver.session() as session:
        for rel_type, rows in rels_by_type.items():
            cypher = f"""
            UNWIND $rows AS row
            MATCH (a {{id: row.source_id}})
            MATCH (b {{id: row.target_id}})
            MERGE (a)-[r:{rel_type}]->(b)
            SET r += row.props
            """

            session.run(cypher, rows=rows).consume()
            print(f"Imported {len(rows)} {rel_type} relationships")

    print(f"Relationship import complete for {graphml_path.name}")

In [28]:
graphmls = [
    "cold_cases3.graphml",
    "cold_cases4.graphml",
    "cold_cases5.graphml",
    "cold_cases6.graphml",
]

for graphml in graphmls:
    import_graphml_relationships(graphml)

Parsed 0 relationships from cold_cases3.graphml
Relationship import complete for cold_cases3.graphml
Parsed 26 relationships from cold_cases4.graphml
Imported 5 ACCOMPANIED_BY relationships
Imported 4 ATTENDED_PARTY_AT relationships
Imported 1 LAST_SEEN_NEAR relationships
Imported 1 VICTIM_IN relationships
Imported 1 LIVED_AT relationships
Imported 2 RESIDENCE_OF relationships
Imported 4 FAMILY_RELATIONSHIP relationships
Imported 1 SUSPECT_IN relationships
Imported 1 ACCUSED_IN relationships
Imported 2 FILED_CIVIL_SUIT_IN relationships
Imported 3 SEIZED_FROM relationships
Imported 1 FOUND_AT relationships
Relationship import complete for cold_cases4.graphml
Parsed 23 relationships from cold_cases5.graphml
Imported 1 VICTIM_IN relationships
Imported 1 LAST_SEEN_AT relationships
Imported 4 FAMILY_RELATIONSHIP relationships
Imported 2 RESIDENCE_OF relationships
Imported 3 FAMILY_INVOLVED_IN relationships
Imported 2 INVESTIGATED_BY relationships
Imported 2 LIVED_IN_OR_WORKED_NEAR relations